In [11]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity






In [12]:
#create synthetic dataset for Recommender system
np.random.seed(42)
n_users = 100
n_items = 50
ratings = np.random.randint(1, 6, size=(n_users, n_items))

# Create a DataFrame for better visualization
ratings_df = pd.DataFrame(ratings, columns=[f'Item_{i}' for i in range(n_items)])
ratings_df['User_ID'] = [f'User_{i}' for i in range(n_users)]
ratings_df.set_index('User_ID', inplace=True)
ratings_df.head()

,Item_0,Item_1,Item_2,Item_3,Item_4,Item_5,Item_6,Item_7,Item_8,Item_9,Item_10,Item_11,Item_12,Item_13,Item_14,Item_15,Item_16,Item_17,Item_18,Item_19,Item_20,Item_21,Item_22,Item_23,Item_24,Item_25,Item_26,Item_27,Item_28,Item_29,Item_30,Item_31,Item_32,Item_33,Item_34,Item_35,Item_36,Item_37,Item_38,Item_39,Item_40,Item_41,Item_42,Item_43,Item_44,Item_45,Item_46,Item_47,Item_48,Item_49
User_ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
User_0,4,5,3,5,5,2,3,3,3,5,4,3,5,2,4,2,4,5,1,4,2,5,4,1,1,3,3,2,4,4,3,4,4,1,3,5,3,5,1,2,4,1,4,2,2,1,2,5,2,4
User_1,4,4,4,5,3,1,4,2,4,2,2,4,5,2,2,4,2,2,4,4,1,5,5,2,5,2,1,4,4,4,5,1,5,5,1,1,1,1,4,3,3,1,3,3,1,3,5,2,2,1
User_2,4,1,4,2,1,5,3,4,3,3,1,3,5,3,1,5,2,3,1,2,2,4,5,3,1,4,5,4,5,5,3,5,4,5,3,3,4,2,2,5,1,5,4,4,4,4,4,3,2,4
User_3,1,1,1,1,3,1,4,5,1,3,3,1,5,1,3,2,4,3,1,4,1,1,2,4,4,2,3,1,5,1,1,3,1,2,2,4,5,1,1,3,2,5,4,2,4,3,3,1,5,4
User_4,2,3,1,1,4,3,5,3,4,4,3,4,3,2,3,3,4,4,1,1,2,1,3,4,1,1,2,2,3,4,2,1,4,4,1,2,1,4,5,5,3,1,1,3,3,3,4,1,4,3


In [14]:
# Apply Non-negative Matrix Factorization (NMF)
nmf_recommender = NMF(n_components=5, init='random', random_state=42)
user_features = nmf_recommender.fit_transform(ratings)
item_features = nmf_recommender.components_

# Calculate cosine similarity between users
user_similarity = cosine_similarity(user_features)



c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\decomposition\_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [24]:
#recommend items for a specific user
user_id = "User_1"

#get index 
user_index = ratings_df.index.get_loc(user_id)

# compute recommendations
recommendations = np.dot(user_similarity[user_index], ratings_df.values)

#remove already rated items 
already_rated = ratings_df.loc[user_id].values > 0
recommendations[already_rated] = -1

#top 5 items
top_items = np.argsort(recommendations)[::-1][:5]

print(f"Top recommended items for {user_id}: {[f'Item_{i}' for i in top_items]}")


Top recommended items for User_1: ['Item_49', 'Item_48', 'Item_47', 'Item_46', 'Item_45']


In [25]:
#function to find similar items usin cosine similarity
def find_similar_items(item_id, item_features, top_n=5):
    item_index = int(item_id.split('_')[1])
    item_similarity = cosine_similarity(item_features.T)
    similar_items = np.argsort(item_similarity[item_index])[::-1][1:top_n+1]
    return [f'Item_{i}' for i in similar_items]



In [26]:
#find the similar items for a specific item
item_id = "Item_10" 
similar_items = find_similar_items(item_id, item_features, top_n=5)
print(f"Similar items to {item_id}: {similar_items}")   


Similar items to Item_10: ['Item_15', 'Item_6', 'Item_4', 'Item_47', 'Item_38']


In [27]:
#function to recommend specific values based on user preferences
def recommend_items_for_user(user_id, user_similarity, ratings_df, top_n=5):
    user_index = ratings_df.index.get_loc(user_id)
    recommendations = np.dot(user_similarity[user_index], ratings_df.values)
    already_rated = ratings_df.loc[user_id].values > 0
    recommendations[already_rated] = -1
    top_items = np.argsort(recommendations)[::-1][:top_n]
    return [f'Item_{i}' for i in top_items]


In [28]:
#recommend items for a specific user
user_id = "User_1"  # Example user ID
recommend_specific_item = recommend_items_for_user(user_id, user_similarity, ratings_df, top_n=5)
print(f"Recommended items for {user_id}: {recommend_specific_item}")


Recommended items for User_1: ['Item_49', 'Item_48', 'Item_47', 'Item_46', 'Item_45']
